# 05 - Pipeline central et logs SQLite

Objectif : préparer le pipeline central qui devra devenir `src/pipeline.py`.

Fichiers concernés : `src/preprocessing.py`, `src/inference.py`, `src/guardrails.py`, `src/database.py`, `api/main.py`, `app/streamlit_app.py`, `eval/run_evaluation.py`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_IMAGES_DIR = DATA_DIR / "sample_images"
SRC_DIR = PROJECT_ROOT / "src"
API_DIR = PROJECT_ROOT / "api"
APP_DIR = PROJECT_ROOT / "app"
EVAL_DIR = PROJECT_ROOT / "eval"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
TESTS_DIR = PROJECT_ROOT / "tests"
OUTPUTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
import json, sqlite3, tempfile, time
import pandas as pd
from datetime import datetime
from src.inference import toy_predict
from src.guardrails import apply_safety_guardrails
try:
    from src.preprocessing import load_image
except Exception:
    load_image = None

DEFAULT_DB_PATH = Path(tempfile.gettempdir()) / "assistant_radio_pipeline_logs.sqlite"

def init_prediction_log_db(db_path):
    conn = sqlite3.connect(db_path)
    conn.execute("CREATE TABLE IF NOT EXISTS prediction_logs (id INTEGER PRIMARY KEY AUTOINCREMENT, case_id TEXT, image_path TEXT NOT NULL, mode TEXT NOT NULL, prediction_json TEXT NOT NULL, predicted_class TEXT, confidence REAL, latency_ms REAL, created_at TEXT NOT NULL)")
    conn.commit(); conn.close()

def insert_prediction_log(db_path, case_id, image_path, mode, prediction, latency_ms):
    init_prediction_log_db(db_path)
    conn = sqlite3.connect(db_path)
    conn.execute("INSERT INTO prediction_logs(case_id,image_path,mode,prediction_json,predicted_class,confidence,latency_ms,created_at) VALUES (?,?,?,?,?,?,?,?)", (case_id, str(image_path), mode, json.dumps(prediction, ensure_ascii=False), prediction.get("predicted_class"), float(prediction.get("confidence", 0.0) or 0.0), float(latency_ms), datetime.now().isoformat(timespec="seconds")))
    conn.commit(); conn.close()

def run_pipeline(image_path, mode="toy", db_path=None, case_id=None):
    db_path = Path(db_path) if db_path else DEFAULT_DB_PATH
    start = time.perf_counter()
    if load_image is not None:
        _ = load_image(image_path)
    inference_mode = "improved" if mode in {"toy", "improved"} else "baseline"
    pred = apply_safety_guardrails(toy_predict(image_path, mode=inference_mode))
    latency_ms = round((time.perf_counter() - start) * 1000, 3)
    pred["pipeline_latency_ms"] = latency_ms
    insert_prediction_log(db_path, case_id, image_path, mode, pred, latency_ms)
    return pred

In [ ]:
db_path = DEFAULT_DB_PATH
pred = run_pipeline(SAMPLE_IMAGES_DIR / "CXR_SYN_002_suspected_opacity.png", mode="toy", db_path=db_path, case_id="CXR_SYN_002")
display(pred)
conn = sqlite3.connect(db_path)
logs_df = pd.read_sql_query("SELECT * FROM prediction_logs ORDER BY id DESC LIMIT 10", conn)
conn.close()
out = OUTPUTS_DIR / "pipeline_logs_preview.csv"
logs_df.to_csv(out, index=False)
print(out)
display(logs_df)

Conclusion : transférer ce pipeline dans `src/pipeline.py`; API, Streamlit et évaluation devront tous appeler `run_pipeline`.